In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.2 MB/s eta 0:00:00a 0:00:01


In [4]:
!git clone https://github.com/abewley/sort.git

Cloning into 'sort'...
remote: Enumerating objects: 208, done.
remote: Counting objects: 100% (49/49), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 208 (delta 45), reused 40 (delta 40), pack-reused 159 (from 1)
Receiving objects: 100% (208/208), 1.20 MiB | 9.71 MiB/s, done.
Resolving deltas: 100% (76/76), done.


In [5]:
import sys
sys.path.append("./sort")

In [6]:
!pip install filterpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 4.5 MB/s eta 0:00:0000:01
  Preparing metadata (setup.py) ... done
  Created wheel for filterpy: filename=filterpy-1.4.5-py3-none-any.whl size=110460 sha256=aa3668c2499922bc66b030b6788e809dea8d96b5827857dbdc406e13773a6c5c
  Stored in directory: /root/.cache/pip/wheels/77/bf/4c/b0c3f4798a0166668752312a67118b27a3cd341e13ac0ae6ee
Successfully built filterpy


In [7]:
import sys
sys.path.insert(0, "/kaggle/working/sort")

In [8]:
sort_path = "/kaggle/working/sort/sort.py"

# Read file
with open(sort_path, "r") as f:
    lines = f.readlines()

# Remove matplotlib-related lines
new_lines = []
for line in lines:
    if "matplotlib" in line or "plt." in line:
        continue
    new_lines.append(line)

# Write cleaned file
with open(sort_path, "w") as f:
    f.writelines(new_lines)

print(" sort.py cleaned (matplotlib removed)")

 sort.py cleaned (matplotlib removed)


In [12]:
import os
os.environ["MPLBACKEND"] = "Agg"

import cv2
import numpy as np
from collections import defaultdict, deque
from ultralytics import YOLO
import importlib.util
import itertools
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ================================================================
#  LOAD SORT
# ================================================================
spec = importlib.util.spec_from_file_location(
    "sort_module", "/kaggle/working/sort/sort.py"
)
sort_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(sort_module)
Sort = sort_module.Sort

# ================================================================
#  MODEL & VIDEO
# ================================================================
model = YOLO("/kaggle/input/datasets/shashwatkumarjha/wajanfinal/best.pt")
cap   = cv2.VideoCapture(
    "/kaggle/input/datasets/shashwatkumarjha/thodisibadiclip/Thodisibadiclip.mp4"
)

FPS        = cap.get(cv2.CAP_PROP_FPS)
if FPS <= 1: FPS = 30
dt_nominal = 1.0 / FPS

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out    = cv2.VideoWriter("/kaggle/working/finalsssss.mp4", fourcc, int(FPS), (960, 540))

tracker = Sort(max_age=20, min_hits=3, iou_threshold=0.3)

# ================================================================
#  PARAMETERS  — change only here
# ================================================================
ALPHA_VEL              = 0.15
ALPHA_ACC              = 0.05
ALPHA_SPEED            = 0.15
TRAJ_FIT_MIN_PTS       = 6
TRAJ_FIT_WINDOW        = 20

DISTANCE_GATE_M        = 40.0   # skip pair evaluation beyond this gap
MIN_APPROACH_DIST_M    = 1.5    # closest-approach threshold → paths must converge here
SIDE_BY_SIDE_LATERAL_M = 1.0   # extra lateral guard for parallel lanes
TTC_MAX_S              = 5.0    # ignore TTC beyond this

FOLLOWING_ANGLE_DEG    = 25.0   # headings within this → REAR-END scenario
FOLLOWING_LONGIT_GAP_M = 1.5

RISK_PERSIST_FRAMES    = 5      # pair must be at-risk N consecutive frames → show alert

# ---- REAR-END thresholds (MTTC) ----
DRAC_CRITICAL          = 3.4    # m/s² AASHTO unsafe
DRAC_WARNING           = 1.0
MTTC_CRITICAL_S        = 1.5    # seconds
MTTC_WARNING_S         = 3.0

# ---- CROSSING thresholds (PET) ----
#   PET < 1 s → CRITICAL  (vehicles nearly collide at the conflict point)
#   PET < 2 s → WARNING
PET_CRITICAL_S         = 1.0
PET_WARNING_S          = 2.0

# ---- World bounds (from homography calibration) ----
WORLD_X_MIN, WORLD_X_MAX = 0.0, 3.0
WORLD_Y_MIN, WORLD_Y_MAX = 0.0, 74.65
GRID_X_STEP = 1.0     # 1 m across road  (used for both display grid & heatmap)
GRID_Y_STEP = 10.0    # 10 m along road

LOG_INTERVAL_FRAMES    = 15     # log one xlsx row per pair per N frames

# ---- Colours ----
COLOR_TRAJ_PATH   = (0, 140, 255)    # orange — extrapolated trajectory
COLOR_GRID        = (50,  50,  50)   # subtle grid lines
COLOR_GRID_LABEL  = (130, 130,  60)  # grid labels
COLOR_AXIS_X      = (180,  60,  60)  # x=0 axis
COLOR_AXIS_Y      = (60,  180,  60)  # y=0 axis

class_names  = {0: "Car", 1: "Person", 2: "2-Wheeler"}
class_colors = {0: (0, 255, 0), 1: (255, 128, 0), 2: (0, 200, 255)}

# ================================================================
#  PER-TRACK STATE
# ================================================================
track_history_world = defaultdict(lambda: deque(maxlen=40))
track_history_px    = defaultdict(lambda: deque(maxlen=40))
track_speed         = defaultdict(float)
velocity_ema        = defaultdict(lambda: np.zeros(2))
accel_ema           = defaultdict(lambda: np.zeros(2))
prev_vel_for_acc    = defaultdict(lambda: np.zeros(2))
track_id_to_class   = {}

risk_persist = defaultdict(int)
stable_alerts = {}

# ================================================================
#  HEATMAP ACCUMULATORS
#
#  We maintain two 2-D numpy grids in world-coordinate space:
#    heatmap_mttc  → rear-end conflict points
#    heatmap_pet   → crossing conflict points
#
#  Each cell corresponds to a (GRID_X_STEP × GRID_Y_STEP) tile.
#  When a conflict is confirmed, we map the closest-approach point
#  p_ca into the appropriate cell and increment it.
# ================================================================
_hm_nx = max(1, int(np.ceil((WORLD_X_MAX - WORLD_X_MIN) / GRID_X_STEP)))
_hm_ny = max(1, int(np.ceil((WORLD_Y_MAX - WORLD_Y_MIN) / GRID_Y_STEP)))

heatmap_mttc = np.zeros((_hm_ny, _hm_nx), dtype=np.float32)   # rows=Y, cols=X
heatmap_pet  = np.zeros((_hm_ny, _hm_nx), dtype=np.float32)

def world_to_heatmap_cell(X, Y):
    """
    Convert a world coordinate (X, Y) to (col, row) indices in the
    heatmap grid.  Clamps to valid range so edge points don't crash.
    """
    col = int((X - WORLD_X_MIN) / GRID_X_STEP)
    row = int((Y - WORLD_Y_MIN) / GRID_Y_STEP)
    col = np.clip(col, 0, _hm_nx - 1)
    row = np.clip(row, 0, _hm_ny - 1)
    return col, row

# ================================================================
#  COLLISION LOG
# ================================================================
collision_log = []
last_logged_frame = defaultdict(lambda: -999)

def fmt_timestamp(seconds):
    h  = int(seconds // 3600)
    m  = int((seconds % 3600) // 60)
    s  = int(seconds % 60)
    ms = int((seconds % 1) * 1000)
    return f"{h:02d}:{m:02d}:{s:02d}.{ms:03d}"

# ================================================================
#  HOMOGRAPHY
# ================================================================
scale_x = 960 / 1920
scale_y = 540 / 1080

img_pts = np.array([
    [972  * scale_x, 216  * scale_y],
    [1080 * scale_x, 216  * scale_y],
    [1650 * scale_x, 1066 * scale_y],
    [540  * scale_x, 1066 * scale_y],
], dtype=np.float32)

world_pts = np.array([
    [0,  74.65],
    [3,  74.65],
    [3,  0],
    [0,  0],
], dtype=np.float32)

H, _  = cv2.findHomography(img_pts, world_pts)
H_inv = np.linalg.inv(H)

_errs = []
for ip, wp in zip(img_pts, world_pts):
    p = H @ np.array([ip[0], ip[1], 1.0]); p /= p[2]
    _errs.append(np.linalg.norm(p[:2] - wp))
print(f"[Homography] max reprojection error: {max(_errs):.4f} m")

# ================================================================
#  COORDINATE UTILITIES
# ================================================================
def pixel_to_world(u, v):
    pt = H @ np.array([u, v, 1.0]); pt /= pt[2]
    return float(pt[0]), float(pt[1])

def world_to_pixel(X, Y):
    pt = H_inv @ np.array([X, Y, 1.0]); pt /= pt[2]
    return int(pt[0]), int(pt[1])

# ================================================================
#  WORLD GRID  (pre-computed once; H is constant)
# ================================================================
_grid_lines  = []
_grid_labels = []

def _build_grid():
    for x in np.arange(WORLD_X_MIN, WORLD_X_MAX + GRID_X_STEP * 0.5, GRID_X_STEP):
        p0 = world_to_pixel(x, WORLD_Y_MIN)
        p1 = world_to_pixel(x, WORLD_Y_MAX)
        is_axis = abs(x - WORLD_X_MIN) < 1e-4
        _grid_lines.append((p0, p1, COLOR_AXIS_Y if is_axis else COLOR_GRID,
                             2 if is_axis else 1))
        lp = world_to_pixel(x, WORLD_Y_MIN + 3.0)
        _grid_labels.append((lp[0] + 3, lp[1], f"x={x:.0f}m"))

    for y in np.arange(WORLD_Y_MIN, WORLD_Y_MAX + GRID_Y_STEP * 0.5, GRID_Y_STEP):
        p0 = world_to_pixel(WORLD_X_MIN, y)
        p1 = world_to_pixel(WORLD_X_MAX, y)
        is_axis = abs(y - WORLD_Y_MIN) < 1e-4
        _grid_lines.append((p0, p1, COLOR_AXIS_X if is_axis else COLOR_GRID,
                             2 if is_axis else 1))
        lp = world_to_pixel(WORLD_X_MIN, y)
        _grid_labels.append((lp[0] + 4, lp[1] - 3, f"{y:.0f}m"))

_build_grid()

def draw_world_grid(frame):
    ov = frame.copy()
    for (p0, p1, color, thick) in _grid_lines:
        cv2.line(ov, p0, p1, color, thick, cv2.LINE_AA)
    cv2.addWeighted(ov, 0.55, frame, 0.45, 0, frame)
    for (lx, ly, txt) in _grid_labels:
        (tw, th), _ = cv2.getTextSize(txt, cv2.FONT_HERSHEY_SIMPLEX, 0.28, 1)
        cv2.rectangle(frame, (lx-1, ly-th-1), (lx+tw+1, ly+2), (10, 10, 10), -1)
        cv2.putText(frame, txt, (lx, ly),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.28, COLOR_GRID_LABEL, 1, cv2.LINE_AA)
    cv2.putText(frame, "-- x-axis (road width, m)",
                (660, 510), cv2.FONT_HERSHEY_SIMPLEX, 0.33, COLOR_AXIS_Y, 1)
    cv2.putText(frame, "-- y-axis (road length, m)",
                (660, 524), cv2.FONT_HERSHEY_SIMPLEX, 0.33, COLOR_AXIS_X, 1)

# ================================================================
#  IoU
# ================================================================
def box_iou(a, b):
    xA, yA = max(a[0], b[0]), max(a[1], b[1])
    xB, yB = min(a[2], b[2]), min(a[3], b[3])
    inter  = max(0, xB - xA) * max(0, yB - yA)
    if inter == 0: return 0.0
    return inter / ((a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter)

# ================================================================
#  MOTION ESTIMATION
# ================================================================
def compute_ema_velocity(tid, dt):
    hist = track_history_world[tid]
    if len(hist) < 2: return np.zeros(2)
    raw = (hist[-1] - hist[-2]) / dt
    v   = ALPHA_VEL * raw + (1.0 - ALPHA_VEL) * velocity_ema[tid]
    velocity_ema[tid] = v
    return v

def fit_regression_velocity(tid):
    """
    Linear regression over last TRAJ_FIT_WINDOW positions.
    Gives a stable direction vector that doesn't flicker between frames.
    Used only for trajectory extrapolation and geometry checks.
    """
    hist = list(track_history_world[tid])
    n    = len(hist)
    if n < TRAJ_FIT_MIN_PTS: return velocity_ema[tid].copy()
    pts = np.array(hist[-TRAJ_FIT_WINDOW:])
    t   = np.arange(len(pts), dtype=float) * dt_nominal
    vx  = np.polyfit(t, pts[:, 0], 1)[0]
    vy  = np.polyfit(t, pts[:, 1], 1)[0]
    return np.array([vx, vy])

def compute_acceleration(tid, vel, dt):
    raw = (vel - prev_vel_for_acc[tid]) / dt
    acc = ALPHA_ACC * raw + (1.0 - ALPHA_ACC) * accel_ema[tid]
    accel_ema[tid]        = acc
    prev_vel_for_acc[tid] = vel
    return acc

# ================================================================
#  TRAJECTORY GEOMETRY
# ================================================================
def closest_approach(p1, v1, p2, v2):
    """
    For two linear trajectories P_i(t) = p_i + v_i*t,
    find the time and distance of closest approach.

        d²(t) = |rel_p + rel_v·t|²
        Minimise → t_ca = -(rel_p · rel_v) / |rel_v|²

    Returns
    -------
    t_ca  : time of closest approach (s); negative → already diverging
    d_ca  : distance at closest approach (m)
    p_ca  : world midpoint at closest approach — this is the CONFLICT POINT
    """
    rel_p = p2 - p1;  rel_v = v2 - v1
    denom = np.dot(rel_v, rel_v)
    if denom < 1e-6:
        return 0.0, float(np.linalg.norm(rel_p)), (p1 + p2) / 2.0
    t_ca  = -np.dot(rel_p, rel_v) / denom
    fp1   = p1 + v1 * t_ca;  fp2 = p2 + v2 * t_ca
    return t_ca, float(np.linalg.norm(fp1 - fp2)), (fp1 + fp2) / 2.0

def lateral_separation(p1, v1, p2):
    s = np.linalg.norm(v1)
    if s < 1e-3: return float(np.linalg.norm(p2 - p1))
    perp = np.array([-v1[1], v1[0]]) / s
    return abs(np.dot(p2 - p1, perp))

def is_following_scenario(v1, v2, p1, p2):
    """
    Returns True when the two vehicles are in a REAR-END configuration:
      • Headings are within FOLLOWING_ANGLE_DEG of each other
      • One is longitudinally ahead of the other along their shared heading
    """
    s1, s2 = np.linalg.norm(v1), np.linalg.norm(v2)
    if s1 < 0.5 or s2 < 0.5: return False
    if np.dot(v1/s1, v2/s2) < np.cos(np.radians(FOLLOWING_ANGLE_DEG)):
        return False
    avg = (v1/s1 + v2/s2);  avg /= (np.linalg.norm(avg) + 1e-9)
    return abs(np.dot(p2 - p1, avg)) > 0.1

# ================================================================
#  SAFETY METRICS
# ================================================================
def compute_drac(p1, p2, v1, v2):
    """
    DRAC = (ΔV)² / S    [Almqvist 1991, Eq. 12]
    ΔV = magnitude of relative velocity vector
    S  = Euclidean gap between vehicles
    Threshold: > 3.4 m/s² is unsafe (AASHTO)
    """
    dv = np.linalg.norm(v2 - v1)
    S  = np.linalg.norm(p2 - p1)
    if S < 1e-3 or dv < 1e-3: return 0.0
    return (dv ** 2) / S


def compute_mttc(p1, v1, a1, p2, v2, a2):
    """
    MTTC — Modified Time to Collision accounting for relative acceleration.

    Solves the gap-closing quadratic along the approach axis:
        s(t) = S − dv·t − ½·da·t²  = 0
        ½·da·t² + dv·t − S = 0

    All scalars are projected onto the unit approach direction d̂
    to preserve sign (closing vs. opening).

    Used only for REAR-END scenarios.
    """
    rel_p = p2 - p1;  dist = np.linalg.norm(rel_p)
    if dist < 1e-3: return np.inf
    d  = rel_p / dist
    S  = np.dot(rel_p, d);   dv = np.dot(v2-v1, d);   da = np.dot(a2-a1, d)
    if abs(da) < 1e-5:
        return (-S / dv) if dv < 0 else np.inf
    disc = dv**2 + 2.0 * da * S
    if disc < 0: return np.inf
    sq    = np.sqrt(disc)
    valid = [t for t in [(-dv+sq)/da, (-dv-sq)/da] if t > 0]
    return float(min(valid)) if valid else np.inf


def compute_pet(p1, v1, p2, v2, p_ca):
    """
    PET — Post Encroachment Time.

    Definition (traffic conflict theory):
        PET = |t₁ - t₂|
    where t_i is the time vehicle i takes to reach the conflict point.

    The conflict point is the closest-approach point p_ca, already
    computed by closest_approach().

    Each vehicle's travel time to p_ca is estimated by projecting
    the vector (p_ca − p_i) onto its own velocity direction:

        t_i = (p_ca − p_i) · v̂_i  /  |v_i|

    Physical meaning
    ─────────────────
    PET measures the temporal gap at the conflict zone:
      • Small PET → both vehicles pass through nearly simultaneously → high risk
      • Large PET → one clears before the other arrives → low risk
    Standard thresholds:
      PET < 1 s → CRITICAL
      PET < 2 s → WARNING

    Used ONLY for CROSSING (non-following) scenarios.
    MTTC is NOT computed for crossing — it assumes 1-D closing motion
    and is not physically meaningful at intersection conflicts.
    """
    def time_to_point(p, v, target):
        speed = np.linalg.norm(v)
        if speed < 0.5:          # nearly stationary → treat as very long time
            return np.inf
        v_hat = v / speed
        proj  = np.dot(target - p, v_hat)
        if proj < 0:             # conflict point is behind the vehicle
            return np.inf
        return proj / speed

    t1 = time_to_point(p1, v1, p_ca)
    t2 = time_to_point(p2, v2, p_ca)

    if t1 == np.inf or t2 == np.inf:
        return np.inf

    return abs(t1 - t2)

# ================================================================
#  RISK TIERING  (scenario-aware)
# ================================================================
def risk_tier_rear_end(TTC, DRAC_val, MTTC_val):
    """
    Rear-end risk uses TTC + DRAC + MTTC.
    All three must indicate risk for CRITICAL.
    """
    if (TTC   < MTTC_CRITICAL_S and
        DRAC_val >= DRAC_CRITICAL and
        MTTC_val < MTTC_CRITICAL_S):
        return "CRITICAL", (0,   0, 255)
    if (TTC   < MTTC_WARNING_S or
        DRAC_val >= DRAC_WARNING or
        MTTC_val < MTTC_WARNING_S):
        return "WARNING",  (0, 140, 255)
    return "MONITOR", (0, 210, 210)


def risk_tier_crossing(PET_val):
    """
    Crossing risk uses PET only.
    PET < 1 s → CRITICAL  (near-simultaneous passage through conflict point)
    PET < 2 s → WARNING
    """
    if PET_val < PET_CRITICAL_S:  return "CRITICAL", (0,   0, 255)
    if PET_val < PET_WARNING_S:   return "WARNING",  (0, 140, 255)
    return "MONITOR", (0, 210, 210)

# ================================================================
#  HEATMAP HELPERS
# ================================================================
def accumulate_heatmap(heatmap, X, Y):
    """Increment the grid cell corresponding to world point (X, Y)."""
    col, row = world_to_heatmap_cell(X, Y)
    heatmap[row, col] += 1.0


def render_heatmap_overlay(heatmap, frame, cmap_name, title, alpha=0.45):
    """
    Overlay a normalised heatmap onto the video frame.

    Steps:
      1. Normalise to [0, 1]
      2. Apply a matplotlib colormap
      3. Resize to frame dimensions (nearest-neighbour to keep grid edges sharp)
      4. Alpha-blend over frame

    The grid is coarse (3×8 cells for this road), so the overlay
    is visibly chunky — this is intentional; it directly mirrors
    the world coordinate grid drawn on the frame.
    """
    if heatmap.max() < 1e-6:
        return   # nothing to show yet

    norm    = heatmap / heatmap.max()
    cmap    = plt.get_cmap(cmap_name)
    colored = (cmap(norm)[:, :, :3] * 255).astype(np.uint8)  # (ny, nx, 3) RGB
    colored_bgr = cv2.cvtColor(colored, cv2.COLOR_RGB2BGR)

    # Resize to frame size (interpolation=NEAREST keeps cells blocky, not blurry)
    h, w = frame.shape[:2]
    colored_bgr = cv2.resize(colored_bgr, (w, h), interpolation=cv2.INTER_NEAREST)

    cv2.addWeighted(colored_bgr, alpha, frame, 1.0 - alpha, 0, frame)

    # Title badge
    cv2.rectangle(frame, (4, 4), (len(title)*8 + 10, 22), (20, 20, 20), -1)
    cv2.putText(frame, title, (7, 17),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)


def save_heatmap_image(heatmap, path, title, cmap_name, world_bounds):
    """
    Save a standalone heatmap figure using matplotlib.
    Axes are in world coordinates (metres).
    Called once after the video loop finishes.
    """
    if heatmap.max() < 1e-6:
        print(f"[Heatmap] {title}: no data, skipping.")
        return

    x_min, x_max, y_min, y_max = world_bounds
    fig, ax = plt.subplots(figsize=(4, 10))
    im = ax.imshow(
        heatmap,
        origin="lower",
        extent=[x_min, x_max, y_min, y_max],
        cmap=cmap_name,
        aspect="auto",
        interpolation="nearest",
    )
    plt.colorbar(im, ax=ax, label="Conflict count")
    ax.set_xlabel("X — road width (m)")
    ax.set_ylabel("Y — road length (m)")
    ax.set_title(title, fontsize=11, fontweight="bold")

    # Draw grid lines over the heatmap
    for x in np.arange(x_min, x_max + GRID_X_STEP, GRID_X_STEP):
        ax.axvline(x, color="white", linewidth=0.4, alpha=0.6)
    for y in np.arange(y_min, y_max + GRID_Y_STEP, GRID_Y_STEP):
        ax.axhline(y, color="white", linewidth=0.4, alpha=0.6)

    # Annotate cells with counts
    for row in range(_hm_ny):
        for col in range(_hm_nx):
            val = int(heatmap[row, col])
            if val > 0:
                cx = x_min + (col + 0.5) * GRID_X_STEP
                cy = y_min + (row + 0.5) * GRID_Y_STEP
                ax.text(cx, cy, str(val), ha="center", va="center",
                        color="white", fontsize=7, fontweight="bold")

    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"[Heatmap] Saved: {path}")

# ================================================================
#  VISUALIZATION HELPERS
# ================================================================
def draw_trajectory(frame, tid, color):
    pts = list(track_history_px[tid])
    for i in range(1, len(pts)):
        alpha = i / len(pts)
        c = tuple(int(ch * alpha) for ch in color)
        cv2.line(frame, pts[i-1], pts[i], c, 2, cv2.LINE_AA)

def draw_extrapolated_path(frame, pos_w, vel_reg, horizon_s, steps=16):
    """Dashed orange future trajectory (regression velocity → stable, no flicker)."""
    prev_px = world_to_pixel(pos_w[0], pos_w[1])
    for s in range(1, steps + 1):
        fw      = pos_w + vel_reg * (horizon_s * s / steps)
        curr_px = world_to_pixel(fw[0], fw[1])
        if s % 2 == 1:
            cv2.line(frame, prev_px, curr_px, COLOR_TRAJ_PATH, 2, cv2.LINE_AA)
        prev_px = curr_px

def draw_hud(frame, alerts):
    if not alerts: return
    h  = 26 + 22 * len(alerts)
    ov = frame.copy()
    cv2.rectangle(ov, (4, 4), (480, h), (15, 15, 15), -1)
    cv2.addWeighted(ov, 0.55, frame, 0.45, 0, frame)
    cv2.putText(frame, "COLLISION RISK MONITOR",
                (9, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.52, (190, 190, 190), 1)
    order = {"CRITICAL": 0, "WARNING": 1, "MONITOR": 2}
    for i, (lvl, col, txt) in enumerate(
            sorted(alerts, key=lambda x: order[x[0]])):
        cv2.putText(frame, txt, (9, 40 + 22*i),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.42, col, 1, cv2.LINE_AA)

# ================================================================
#  XLSX WRITER
# ================================================================
def write_collision_xlsx(log, path):
    wb  = Workbook()
    thin = Side(style="thin", color="AAAAAA")
    bdr  = Border(left=thin, right=thin, top=thin, bottom=thin)
    hdr_fill  = PatternFill("solid", start_color="1F3864")
    hdr_font  = Font(bold=True, color="FFFFFF", name="Arial", size=10)
    hdr_align = Alignment(horizontal="center", vertical="center", wrap_text=True)
    row_font  = Font(name="Arial", size=9)
    ctr       = Alignment(horizontal="center")
    fills     = {
        "CRITICAL": PatternFill("solid", start_color="FFCCCC"),
        "WARNING":  PatternFill("solid", start_color="FFE5CC"),
        "MONITOR":  PatternFill("solid", start_color="FFFFCC"),
    }

    # ── Sheet 1: full event log ────────────────────────────────────
    ws1 = wb.active;  ws1.title = "Collision Event Log"

    headers = [
        "Frame", "Timestamp", "Video Time (s)",
        "ID_1", "Class_1", "ID_2", "Class_2",
        "Scenario", "Risk Level",
        "TTC (s)", "DRAC (m/s²)",
        "MTTC (s) [rear-end]", "PET (s) [crossing]",
        "Distance (m)", "Rel Speed (m/s)",
    ]
    ws1.row_dimensions[1].height = 30
    for ci, h in enumerate(headers, 1):
        c = ws1.cell(row=1, column=ci, value=h)
        c.font = hdr_font;  c.fill = hdr_fill
        c.alignment = hdr_align;  c.border = bdr

    for ri, ev in enumerate(log, 2):
        def _fmt(v): return round(v, 3) if v < 999 else "∞"
        vals = [
            ev["frame"], ev["timestamp"], round(ev["time_s"], 3),
            ev["id1"], ev["cls1"], ev["id2"], ev["cls2"],
            ev["scenario"], ev["level"],
            round(ev["ttc"], 3), round(ev["drac"], 3),
            _fmt(ev["mttc"]),   # filled only for REAR-END
            _fmt(ev["pet"]),    # filled only for CROSSING
            round(ev["dist"], 2), round(ev["rel_speed"], 2),
        ]
        fill = fills.get(ev["level"], fills["MONITOR"])
        for ci, v in enumerate(vals, 1):
            c = ws1.cell(row=ri, column=ci, value=v)
            c.fill = fill;  c.font = row_font
            c.alignment = ctr;  c.border = bdr

    for i, w in enumerate(
            [8,14,14,6,10,6,10,12,10,10,12,14,14,12,14], 1):
        ws1.column_dimensions[get_column_letter(i)].width = w
    ws1.freeze_panes = "A2"

    # ── Sheet 2: pair summary ──────────────────────────────────────
    ws2 = wb.create_sheet("Pair Summary")
    s2h = ["ID_1","Class_1","ID_2","Class_2","Scenario",
           "# Frames","Min TTC (s)","Max DRAC (m/s²)",
           "Min MTTC (s)","Min PET (s)",
           "First Seen","Last Seen","Peak Level"]
    for ci, h in enumerate(s2h, 1):
        c = ws2.cell(row=1, column=ci, value=h)
        c.font = hdr_font;  c.fill = hdr_fill
        c.alignment = hdr_align;  c.border = bdr
    ws2.row_dimensions[1].height = 30

    from collections import defaultdict as dd
    pairs = dd(lambda: dict(count=0, ttcs=[], dracs=[], mttcs=[], pets=[],
                             first="", last="", levels=[],
                             cls1="", cls2="", scenario=""))
    for ev in log:
        k = (ev["id1"], ev["id2"]);  p = pairs[k]
        p["count"] += 1
        p["ttcs"].append(ev["ttc"]);  p["dracs"].append(ev["drac"])
        if ev["mttc"] < 999: p["mttcs"].append(ev["mttc"])
        if ev["pet"]  < 999: p["pets"].append(ev["pet"])
        if not p["first"]: p["first"] = ev["timestamp"]
        p["last"] = ev["timestamp"];  p["levels"].append(ev["level"])
        p["cls1"] = ev["cls1"];       p["cls2"] = ev["cls2"]
        p["scenario"] = ev["scenario"]

    order_map = {"CRITICAL": 0, "WARNING": 1, "MONITOR": 2}
    for ri, ((id1, id2), p) in enumerate(pairs.items(), 2):
        peak     = min(p["levels"], key=lambda x: order_map[x])
        min_mttc = round(min(p["mttcs"]), 3) if p["mttcs"] else "∞"
        min_pet  = round(min(p["pets"]),  3) if p["pets"]  else "∞"
        vals = [id1, p["cls1"], id2, p["cls2"], p["scenario"],
                p["count"], round(min(p["ttcs"]),3), round(max(p["dracs"]),3),
                min_mttc, min_pet, p["first"], p["last"], peak]
        fill = fills.get(peak, fills["MONITOR"])
        for ci, v in enumerate(vals, 1):
            c = ws2.cell(row=ri, column=ci, value=v)
            c.fill = fill;  c.font = row_font
            c.alignment = ctr;  c.border = bdr

    for i, w in enumerate(
            [6,10,6,10,12,10,12,14,12,12,14,14,12], 1):
        ws2.column_dimensions[get_column_letter(i)].width = w
    ws2.freeze_panes = "A2"

    wb.save(path)
    print(f"[XLSX] {len(log)} events, {len(pairs)} pairs → {path}")

# ================================================================
#  MAIN LOOP
# ================================================================
prev_ms   = cap.get(cv2.CAP_PROP_POS_MSEC)
frame_idx = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break

    curr_ms = cap.get(cv2.CAP_PROP_POS_MSEC)
    dt      = (curr_ms - prev_ms) / 1000.0
    if dt <= 0 or dt > 0.5: dt = dt_nominal
    prev_ms   = curr_ms
    frame_idx += 1
    time_s    = frame_idx / FPS

    frame = cv2.resize(frame, (960, 540))

    # Draw world grid first (sits under everything)
    draw_world_grid(frame)

    # ── ROI ──────────────────────────────────────────────────────
    roi_pts = np.array([
        [int(972  * scale_x), int(216  * scale_y)],
        [int(1080 * scale_x), int(216  * scale_y)],
        [int(1650 * scale_x), int(1066 * scale_y)],
        [int(540  * scale_x), int(1066 * scale_y)],
    ], dtype=np.int32)
    mask         = np.zeros(frame.shape[:2], dtype=np.uint8)
    cv2.fillPoly(mask, [roi_pts], 255)
    frame_masked = cv2.bitwise_and(frame, frame, mask=mask)
    cv2.polylines(frame, [roi_pts], True, (70, 70, 70), 1)

    # ── Detection ────────────────────────────────────────────────
    results     = model(frame_masked, verbose=False)[0]
    detections  = [];  det_classes = []
    for b in results.boxes:
        cls, conf = int(b.cls[0]), float(b.conf[0])
        if cls not in [0, 1, 2] or conf < 0.4: continue
        x1, y1, x2, y2 = b.xyxy[0].cpu().numpy()
        detections.append([x1, y1, x2, y2, conf]);  det_classes.append(cls)

    det_arr = np.array(detections) if detections else np.empty((0, 5))
    tracks  = tracker.update(det_arr)

    # ── Class assignment ─────────────────────────────────────────
    for t in tracks:
        tx1, ty1, tx2, ty2, tid = t;  tid = int(tid)
        if tid in track_id_to_class: continue
        best_iou, best_cls = 0.0, -1
        for det, cls in zip(detections, det_classes):
            ov = box_iou([tx1, ty1, tx2, ty2], det[:4])
            if ov > best_iou: best_iou, best_cls = ov, cls
        if best_iou > 0.3 and best_cls >= 0:
            track_id_to_class[tid] = best_cls

    # ── Vehicle state ─────────────────────────────────────────────
    vehicles = {}
    for t in tracks:
        x1, y1, x2, y2, tid = t;  tid = int(tid)
        u, v = (x1 + x2) / 2.0, y2
        X, Y = pixel_to_world(u, v)

        hw = track_history_world[tid]
        if len(hw) > 0 and np.linalg.norm(np.array([X, Y]) - hw[-1]) > 15.0:
            continue
        hw.append(np.array([X, Y]))
        track_history_px[tid].append((int(u), int(v)))

        vel_ema = compute_ema_velocity(tid, dt)
        vel_reg = fit_regression_velocity(tid)
        acc     = compute_acceleration(tid, vel_ema, dt)

        cls   = track_id_to_class.get(tid, 0)
        color = class_colors[cls];  label = class_names[cls]

        raw_sp = np.linalg.norm(vel_ema) * 3.6
        speed  = ALPHA_SPEED * raw_sp + (1.0 - ALPHA_SPEED) * track_speed[tid]
        track_speed[tid] = speed

        vehicles[tid] = {
            "pos": np.array([X, Y]), "vel_ema": vel_ema,
            "vel_reg": vel_reg, "acc": acc,
            "px": (int(u), int(v)), "cls": cls, "speed": speed, "color": color,
        }

        cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), color, 2)
        cv2.putText(frame, f"{label} ID:{tid}  {speed:.1f}km/h",
                    (int(x1), int(y1)-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.43, color, 1, cv2.LINE_AA)
        draw_trajectory(frame, tid, color)
        draw_extrapolated_path(frame, np.array([X, Y]), vel_reg, TTC_MAX_S)

    # ================================================================
    #  COLLISION PREDICTION
    #
    #  Flow:
    #  1. Distance gate
    #  2. Closest approach → conflict point p_ca, time t_ca, gap d_ca
    #  3. Time validity   (0 < t_ca ≤ TTC_MAX_S)
    #  4. Gap check       (d_ca ≤ MIN_APPROACH_DIST_M)
    #  5. Lateral / following classification
    #  6a. REAR-END  → compute DRAC + MTTC → risk_tier_rear_end()
    #  6b. CROSSING  → compute PET         → risk_tier_crossing()
    #  7. Persist filter (anti-flicker)
    #  8. Heatmap accumulation
    # ================================================================
    current_at_risk = set()

    for (id1, veh1), (id2, veh2) in itertools.combinations(vehicles.items(), 2):
        pair_key = (min(id1, id2), max(id1, id2))

        p1, v1r = veh1["pos"], veh1["vel_reg"]
        p2, v2r = veh2["pos"], veh2["vel_reg"]
        v1e, a1 = veh1["vel_ema"], veh1["acc"]
        v2e, a2 = veh2["vel_ema"], veh2["acc"]

        # ── 1. Distance gate ──────────────────────────────────────
        dist_now = np.linalg.norm(p2 - p1)
        if dist_now > DISTANCE_GATE_M: continue

        # ── 2. Closest approach ───────────────────────────────────
        t_ca, d_ca, p_ca = closest_approach(p1, v1r, p2, v2r)

        # ── 3. Time validity ──────────────────────────────────────
        if t_ca <= 0 or t_ca > TTC_MAX_S:
            risk_persist[pair_key] = 0;  continue

        # ── 4. Trajectory gap check ───────────────────────────────
        if d_ca > MIN_APPROACH_DIST_M:
            risk_persist[pair_key] = 0;  continue

        # ── 5. Scenario classification ────────────────────────────
        lat_sep   = lateral_separation(p1, v1r, p2)
        following = is_following_scenario(v1r, v2r, p1, p2)

        if not following and lat_sep > SIDE_BY_SIDE_LATERAL_M:
            risk_persist[pair_key] = 0;  continue

        if following:
            avg_dir = v1r + v2r;  anorm = np.linalg.norm(avg_dir)
            if anorm > 1e-3:
                avg_dir /= anorm
                if abs(np.dot(p2 - p1, avg_dir)) > FOLLOWING_LONGIT_GAP_M * 3:
                    risk_persist[pair_key] = 0;  continue

        # ── 6. Scenario-specific metrics ──────────────────────────
        # EMA velocity used for metrics (more reactive than regression)
        DRAC_val = compute_drac(p1, p2, v1e, v2e)
        scenario = "REAR-END" if following else "CROSSING"

        if following:
            # ── REAR-END: TTC + DRAC + MTTC ──────────────────────
            MTTC_val = compute_mttc(p1, v1e, a1, p2, v2e, a2)
            PET_val  = np.inf      # not applicable for rear-end
            level, alert_color = risk_tier_rear_end(t_ca, DRAC_val, MTTC_val)
            metric_str = (f"TTC:{t_ca:.2f}s  DRAC:{DRAC_val:.2f}  "
                          f"MTTC:{'∞' if MTTC_val>999 else f'{MTTC_val:.2f}'}s")
        else:
            # ── CROSSING: PET only ────────────────────────────────
            PET_val  = compute_pet(p1, v1r, p2, v2r, p_ca)
            # Use regression velocity for PET (stable direction matters here)
            MTTC_val = np.inf      # not applicable for crossing
            level, alert_color = risk_tier_crossing(PET_val)
            metric_str = (f"TTC:{t_ca:.2f}s  "
                          f"PET:{'∞' if PET_val>999 else f'{PET_val:.2f}'}s  "
                          f"DRAC:{DRAC_val:.2f}")

        # ── 7. Persist filter ─────────────────────────────────────
        current_at_risk.add(pair_key)
        risk_persist[pair_key] += 1
        if risk_persist[pair_key] < RISK_PERSIST_FRAMES: continue

        # ── 8. Heatmap accumulation ───────────────────────────────
        if following:
            # Only count MTTC if it is less than 1.5 seconds (critical rear-end conflicts)
            if MTTC_val < 1.5:
                accumulate_heatmap(heatmap_mttc, p_ca[0], p_ca[1])
        else:
            # Use research-based PET threshold (Allen et al., 1978)
            # Only count unsafe crossing conflicts (PET < 2.0 s)
            if PET_val < 2.0:
                accumulate_heatmap(heatmap_pet, p_ca[0], p_ca[1])
        # ── Store stable alert ────────────────────────────────────
        stable_alerts[pair_key] = {
            "level": level, "color": alert_color,
            "TTC": t_ca, "DRAC": DRAC_val,
            "MTTC": MTTC_val, "PET": PET_val,
            "p_ca": p_ca, "id1": id1, "id2": id2,
            "scenario": scenario, "metric_str": metric_str,
        }

        # ── Log entry ─────────────────────────────────────────────
        if frame_idx - last_logged_frame[pair_key] >= LOG_INTERVAL_FRAMES:
            last_logged_frame[pair_key] = frame_idx
            collision_log.append({
                "frame": frame_idx, "timestamp": fmt_timestamp(time_s),
                "time_s": time_s, "id1": id1, "cls1": class_names[veh1["cls"]],
                "id2": id2, "cls2": class_names[veh2["cls"]],
                "scenario": scenario, "level": level,
                "ttc": t_ca, "drac": DRAC_val,
                "mttc": MTTC_val, "pet": PET_val,
                "dist": dist_now, "rel_speed": np.linalg.norm(v2e - v1e),
            })

    # Expire stale alerts
    for pk in list(stable_alerts.keys()):
        if pk not in current_at_risk:
            risk_persist[pk] = 0;  del stable_alerts[pk]

    # ── Draw alerts ───────────────────────────────────────────────
    hud_rows = []
    for pk, al in stable_alerts.items():
        id1, id2     = al["id1"], al["id2"]
        alert_color  = al["color"]
        cpx          = world_to_pixel(al["p_ca"][0], al["p_ca"][1])

        cv2.circle(frame, cpx, 9,  alert_color, -1)
        cv2.circle(frame, cpx, 14, alert_color, 2)

        if id1 in vehicles and id2 in vehicles:
            cv2.line(frame, vehicles[id1]["px"], vehicles[id2]["px"],
                     alert_color, 1, cv2.LINE_AA)

        hud_rows.append((
            al["level"], alert_color,
            f"[{al['level']}|{al['scenario']}] ID{id1}↔ID{id2}  "
            + al["metric_str"]
        ))

    draw_hud(frame, hud_rows)

    cv2.putText(
        frame,
        f"Frame:{frame_idx}  {fmt_timestamp(time_s)}  Tracks:{len(vehicles)}",
        (630, 537), cv2.FONT_HERSHEY_SIMPLEX, 0.34, (130, 130, 130), 1,
    )
    out.write(frame)

# ================================================================
#  POST-PROCESSING  (after video loop)
# ================================================================
cap.release()
out.release()

# ── XLSX ──────────────────────────────────────────────────────────
xlsx_path = "/kaggle/working/collision_logsssss.xlsx"
if collision_log:
    write_collision_xlsx(collision_log, xlsx_path)
else:
    print("[XLSX] No collision events detected.")

# ── Heatmap images ────────────────────────────────────────────────
bounds = (WORLD_X_MIN, WORLD_X_MAX, WORLD_Y_MIN, WORLD_Y_MAX)

save_heatmap_image(
    heatmap_mttc,
    "/kaggle/working/heatmap_mttc_rearendsssss.png",
    "MTTC Heatmap — Rear-End Conflicts",
    "Reds",
    bounds,
)

save_heatmap_image(
    heatmap_pet,
    "/kaggle/working/heatmap_pet_crossingssss.png",
    "PET Heatmap — Crossing Conflicts",
    "Blues",
    bounds,
)

# ── Side-by-side summary figure ───────────────────────────────────
if heatmap_mttc.max() > 0 or heatmap_pet.max() > 0:
    fig, axes = plt.subplots(1, 2, figsize=(10, 10))
    titles  = ["MTTC — Rear-End Conflicts", "PET — Crossing Conflicts"]
    cmaps   = ["Reds", "Blues"]
    hmaps   = [heatmap_mttc, heatmap_pet]
    for ax, hm, title, cmap in zip(axes, hmaps, titles, cmaps):
        im = ax.imshow(
            hm if hm.max() > 0 else np.zeros_like(hm),
            origin="lower",
            extent=[WORLD_X_MIN, WORLD_X_MAX, WORLD_Y_MIN, WORLD_Y_MAX],
            cmap=cmap, aspect="auto", interpolation="nearest",
        )
        plt.colorbar(im, ax=ax, label="Conflict count")
        ax.set_xlabel("X — road width (m)")
        ax.set_ylabel("Y — road length (m)")
        ax.set_title(title, fontsize=10, fontweight="bold")
        for x in np.arange(WORLD_X_MIN, WORLD_X_MAX + GRID_X_STEP, GRID_X_STEP):
            ax.axvline(x, color="white", lw=0.4, alpha=0.5)
        for y in np.arange(WORLD_Y_MIN, WORLD_Y_MAX + GRID_Y_STEP, GRID_Y_STEP):
            ax.axhline(y, color="white", lw=0.4, alpha=0.5)
        for row in range(_hm_ny):
            for col in range(_hm_nx):
                val = int(hm[row, col])
                if val > 0:
                    cx = WORLD_X_MIN + (col + 0.5) * GRID_X_STEP
                    cy = WORLD_Y_MIN + (row + 0.5) * GRID_Y_STEP
                    ax.text(cx, cy, str(val), ha="center", va="center",
                            color="white", fontsize=8, fontweight="bold")
    plt.suptitle("Traffic Conflict Heatmaps", fontsize=13, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.savefig("/kaggle/working/heatmap_combinednew.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("[Heatmap] Combined figure → /kaggle/working/heatmap_combined.png")

print("\n✅  All outputs:")
print("    Video          → /kaggle/working/final.mp4")
print("    Collision log  → /kaggle/working/collision_log.xlsx")
print("    Heatmap MTTC   → /kaggle/working/heatmap_mttc_rearend.png")
print("    Heatmap PET    → /kaggle/working/heatmap_pet_crossing.png")
print("    Heatmap combo  → /kaggle/working/heatmap_combined.png")

[Homography] max reprojection error: 0.0000 m
[XLSX] 151 events, 75 pairs → /kaggle/working/collision_logsssss.xlsx
[Heatmap] Saved: /kaggle/working/heatmap_mttc_rearendsssss.png
[Heatmap] Saved: /kaggle/working/heatmap_pet_crossingssss.png
[Heatmap] Combined figure → /kaggle/working/heatmap_combined.png

✅  All outputs:
    Video          → /kaggle/working/final.mp4
    Collision log  → /kaggle/working/collision_log.xlsx
    Heatmap MTTC   → /kaggle/working/heatmap_mttc_rearend.png
    Heatmap PET    → /kaggle/working/heatmap_pet_crossing.png
    Heatmap combo  → /kaggle/working/heatmap_combined.png
